# Bank Marketing (PATE-GAN) 

Author: Ilse Harmers \
Last updated: February 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf

In [ ]:
# Importing train set.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")
print(bank_train.columns.to_list())
bank_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_train["age"].min(), upper = bank_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # job
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # default
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-8000) + 1)), upper = np.log(102000 + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # housing
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # loan
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # contact
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # day
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # month
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4900 + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_train["campaign"].min(), upper = 60,
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["pdays"].min()) + 1)), upper = np.log(870 + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(280 + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # poutcome
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # y
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/bank_train.shape[0]))
print(delta)

In [ ]:
# Synthesizing the dataset with PATE-GAN. 
synth = Synthesizer.create('pategan', epsilon = 1.0, delta = delta, plot_losses = True)
synth.fit(bank_train, transformer = tt, preprocessor_eps = 0.0)

In [ ]:
# Generating a synthetic dataset with the same amount of rows as the original training dataset.
sample = synth.sample(bank_train.shape[0])

In [ ]:
sample.describe()

In [ ]:
# Saving the synthetic dataset as a csv file.
sample.to_csv("./synthetic-datasets_PATE-GAN/bank-marketing/epsi_1/run5/sample3.csv", index = False)